In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Load the CSV file using the path (adjust filename if needed)
# Usually the download returns a folder, so you need the full file path
csv_file = path + "/Q3_data.csv"  # Make sure the CSV name matches exactly
df = pd.read_csv(csv_file)


In [ ]:
# Task 2: Write your code here:
# Inspect the first few rows, few is ambguis so i will choose 5
print("First 5 rows of the dataset:")
print(df.head())


In [ ]:
# Task 3: Write your code here:
# Check dataset info (column types, non-null counts)
print("\nDataset information:")
print(df.info())


In [ ]:
# Task 4: Write your code here:
# Show summary statistics for numerical columns
print("\nStatistical description of numerical features:")
print(df.describe())


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Handle missing values
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Fill numerical columns with median
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical columns with mode if any exist
if len(cat_cols) > 0:
    df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])


In [ ]:
# Task 2: Write your code here:


# Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:

# Encode categorical variables if needed
if len(cat_cols) > 0:
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
# Feature scaling for numerical features
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
# Task 5: Write your code here:

# Check target imbalance
target_col = 'Target'  # replace with actual target column name if different
target_counts = df[target_col].value_counts()
print("Target value counts:")
print(target_counts)

if len(target_counts) >= 2:
    imbalance_ratio = target_counts.iloc[0] / target_counts.iloc[1]
    if imbalance_ratio > 2 or imbalance_ratio < 0.5:
        print("The target is imbalanced.")
    else:
        print("The target is fairly balanced.")
else:
    print("Only one unique target value found, cannot compute imbalance.")

In [ ]:
# Task 1: Write your code here:

# Task 1: Split dataset into features and target
X = df.drop('Target', axis=1)  # Replace 'Target' with your actual target column name
y = df['Target']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


#  Initialize K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Train CatBoostRegressor and evaluate
mse_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_seed=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

# Print averaged MSE across folds
avg_mse = sum(mse_scores) / len(mse_scores)
print(f"Averaged MSE across {n_splits} folds: {avg_mse:.4f}")


In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt
import pandas as pd

# Get feature importance from your trained CatBoost model
feature_importances = model.get_feature_importance()
feature_names = X.columns

# Create a DataFrame
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(fi_df['Feature'], fi_df['Importance'], color='skyblue')
plt.xticks(rotation=90)
plt.title("Feature Importance from CatBoost")
plt.show()



In [ ]:
# Task 2: Write your code here:

# The most important feature
golden_feature = fi_df.iloc[0]['Feature']
print("Golden Feature:", golden_feature)


In [ ]:
# Task Bonus: Retrain with Golden Feature Only
X_golden = X[[golden_feature]]  # Keep only the golden feature
print("Shape of new X with golden feature only:", X_golden.shape)

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
golden_mse_scores = []

for train_index, test_index in kf.split(X_golden):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train CatBoostRegressor on single feature
    model_golden = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_seed=42
    )
    model_golden.fit(X_train, y_train)

    # Evaluate
    y_pred = model_golden.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    golden_mse_scores.append(mse)

# Average MSE with golden feature
avg_golden_mse = sum(golden_mse_scores) / len(golden_mse_scores)
avg_golden_rmse = avg_golden_mse ** 0.5

print(f"Average MSE with Golden Feature only: {avg_golden_mse:.4f}")
print(f"Average RMSE with Golden Feature only: {avg_golden_rmse:.4f}")

# Compare with full model metrics
print(f"Average MSE with Full Model: {avg_mse:.4f}")
print(f"Average RMSE with Full Model: {avg_rmse:.4f}")
